# AIS 자료 기초 통계 점검

입력 CSV, Parquet 또는 GeoPackage 자료의 크기, 결측치, 중복, 수치형 요약 통계, 선종별 빈도와 시간 범위를 확인한다. CSV와 GeoPackage는 기본적으로 앞에서 최대 1,000,000행만 읽는다. 이 앞부분 표본은 무작위 표본이 아니므로 전국 데이터의 대표 통계로 해석하지 않는다. 최종 통계에는 지역 추출한 전체 자료를 사용한다. GeoPackage 입력은 geopandas가 필요하다.

입력 자료가 없는 상태에서도 셀을 실행할 수 있도록 경로를 지정하면 파일 확인 단계에서 안내 메시지를 낸다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북 기준 상대 경로. 지역 추출 자료에 맞게 변경한다.
INPUT_PATH = Path('../interim/AISVesselTracks2024.gpkg')
CSV_SAMPLE_ROWS = 1_000_000
GEOPACKAGE_SAMPLE_ROWS = 1_000_000  # None으로 바꾸면 선택 레이어 전체를 읽는다. 지역 추출 자료에만 권장
TOP_CATEGORY_COUNT = 20

input_path = INPUT_PATH.resolve()
print(f'입력 파일: {input_path}')

In [ ]:
if not input_path.exists():
    print('입력 파일이 아직 없습니다. INPUT_PATH를 실제 CSV/Parquet 경로로 바꾼 뒤 다시 실행하세요.')
    df = None
elif input_path.suffix.lower() in {'.parquet', '.pq'}:
    df = pd.read_parquet(input_path)
elif input_path.suffix.lower() == '.csv':
    df = pd.read_csv(input_path, nrows=CSV_SAMPLE_ROWS)
    print(f'CSV는 첫 {CSV_SAMPLE_ROWS:,}행까지 읽었습니다.')
elif input_path.suffix.lower() == '.gpkg':
    import geopandas as gpd
    layers = gpd.list_layers(input_path)
    display(layers)
    GPKG_LAYER = layers.iloc[0]['name']  # 레이어를 확인한 뒤 필요하면 수정한다.
    row_slice = None if GEOPACKAGE_SAMPLE_ROWS is None else slice(0, GEOPACKAGE_SAMPLE_ROWS)
    df = gpd.read_file(input_path, layer=GPKG_LAYER, rows=row_slice)
    if GEOPACKAGE_SAMPLE_ROWS is not None:
        print(f'GeoPackage {GPKG_LAYER!r}의 첫 {GEOPACKAGE_SAMPLE_ROWS:,}개 피처를 읽었습니다.')
else:
    raise ValueError(f'지원하지 않는 파일 형식: {input_path.suffix}')

if df is not None:
    print(f'행 수: {len(df):,} / 열 수: {len(df.columns):,}')
    display(df.head())
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'non_null': df.notna().sum(), 'missing': df.isna().sum()}))

## 수치형 변수 요약

In [ ]:
if df is not None:
    numeric = df.select_dtypes(include=[np.number])
    if numeric.empty:
        print('수치형 열이 없습니다.')
    else:
        summary = numeric.describe(percentiles=[0.25, 0.5, 0.75]).T
        summary['missing'] = numeric.isna().sum()
        summary['missing_rate'] = numeric.isna().mean()
        display(summary)

## 결측치와 완전 중복 행

In [ ]:
if df is not None:
    missing = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_rate': df.isna().mean(),
    }).sort_values('missing_count', ascending=False)
    display(missing[missing['missing_count'] > 0])
    print(f'완전 중복 행: {df.duplicated().sum():,}')

## 선종 및 주요 범주형 변수

In [ ]:
if df is not None:
    preferred_columns = ['VesselGroup', 'VesselType', 'ship_type', 'ShipType', 'Status']
    category_columns = [c for c in preferred_columns if c in df.columns]
    if not category_columns:
        category_columns = [
            c for c in df.select_dtypes(include=['object', 'category', 'string']).columns
            if c.lower() not in {'geometry', 'wkb_geometry'}
        ][:5]
    if not category_columns:
        print('요약할 범주형 열이 없습니다.')
    for column in category_columns:
        print(f'[{column}]')
        display(df[column].value_counts(dropna=False).head(TOP_CATEGORY_COUNT).rename('count').to_frame())

## 시간 범위와 MMSI 고유값 점검

In [ ]:
if df is not None:
    time_candidates = ['TrackStartTime', 'BaseDateTime', 'start_time', 'StartTime']
    time_column = next((c for c in time_candidates if c in df.columns), None)
    if time_column is None:
        print(f'시간 열을 자동으로 찾지 못했습니다. 열 목록: {list(df.columns)}')
    else:
        times = pd.to_datetime(df[time_column], errors='coerce', utc=True)
        print(f'시간 열: {time_column}')
        print(f'파싱된 시각 범위(UTC): {times.min()} ~ {times.max()}')
        print(f'시간 파싱 실패: {times.isna().sum():,}')

    mmsi_column = next((c for c in ['MMSI', 'mmsi'] if c in df.columns), None)
    if mmsi_column is not None:
        print(f'MMSI 고유 개수: {df[mmsi_column].nunique(dropna=True):,}')
        print(f'MMSI 결측 개수: {df[mmsi_column].isna().sum():,}')